In [1]:
import os
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Jupyter ejecutado en Windows: PostgreSQL local en el puerto 5432.
PG_HOST = os.getenv("PGHOST", "localhost")
PG_PORT = int(os.getenv("PGPORT", "5432"))
PG_DATABASE = os.getenv("PGDATABASE", "northwind")
PG_USER = os.getenv("PGUSER", "postgres")
PG_PASSWORD = os.getenv("PGPASSWORD") or getpass("Contraseña de PostgreSQL: ")

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
 )

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

with engine.connect() as connection:
    database_info = connection.execute(
        text("SELECT current_database() AS database, current_user AS user, inet_server_port() AS port")
    ).mappings().one()

print(f"Conectado a {database_info['database']} como {database_info['user']} en el puerto {database_info['port']}")

Conectado a northwind como postgres en el puerto 5432


In [2]:
CLEAN_TABLES = [
    "categories_clean",
    "customers_clean",
    "order_details_clean",
    "orders_clean",
    "products_clean",
]

def load_table(table_name):
    return pd.read_sql_table(table_name, con=engine)

# Carga las tablas depuradas para análisis detallado.
clean_data = {table_name: load_table(table_name) for table_name in CLEAN_TABLES}



for table_name, data_frame in {**clean_data}.items():
    print(f"{table_name}: {data_frame.shape[0]:,} filas x {data_frame.shape[1]} columnas")

categories_clean: 8 filas x 5 columnas
customers_clean: 91 filas x 13 columnas
order_details_clean: 2,155 filas x 7 columnas
orders_clean: 830 filas x 16 columnas
products_clean: 77 filas x 12 columnas
